# Grammar of Lark
- R ::= ∅, ε, a, r1r2, r1|r2, r*, (r1), (?=r1), (?<=r1), (?!r1), (?<!r1)

In [1]:
from enum import Enum, auto
"""
definition of BNF
S -> E
E -> ({E}'|'T) | T
T -> ({T} R) | R
R -> ({R}'*') | A
A -> '(?=' E ')' | '(?<=' E ')' 
     | '(?!' E ')' | '(?<!' E ')' | '(' E ')' 
     | '∅' | 'ε' | ESCAPED | LITERAL
"""

REGEX_GRAMMAR = r"""
?start: alternation

?alternation: alternation "|" concatenation -> make_or
            | concatenation

?concatenation: concatenation repetition -> make_seq
              | repetition

?repetition: repetition "*" -> make_star
           | atom

?atom: "(?=" alternation ")" -> positive_lookahead
     | "(?<=" alternation ")" -> positive_lookbehind
     | "(?!" alternation ")" -> negative_lookahead
     | "(?<!" alternation ")" -> negative_lookbehind
     | "(" alternation ")" -> group
     | "∅" -> empty_set
     | "ε" -> epsilon
     | ESCAPED -> escaped
     | LITERAL -> literal

ESCAPED: /\\[^\n]/
LITERAL: /[^|*()\\∅ε]/
"""

# AstNode
definiton of ast

In [2]:
class AstNode:
    pass
class OrAstNode(AstNode):
    def __init__(self, left, right):
        self.left = left
        self.right = right
class SeqAstNode(AstNode):
    def __init__(self, left, right):
        self.left = left
        self.right = right
class StarAstNode(AstNode):
    def __init__(self, left):
        self.left = left
class LiteralCharacterAstNode(AstNode):
    def __init__(self, char):
        self.char = char
class EmptySetAstNode(AstNode):
    def __init__(self):
        pass
class EpsilonAstNode(AstNode):
    def __init__(self):
        pass
class PositiveLookaheadAstNode(AstNode):
    def __init__(self, left):
        self.left = left
class PositiveLookbehindAstNode(AstNode):
    def __init__(self, left):
        self.left = left
class NegativeLookaheadAstNode(AstNode):
    def __init__(self, left):
        self.left = left
class NegativeLookbehindAstNode(AstNode):
    def __init__(self, left):
        self.left = left
class OraclePolarity(Enum):
    POSITIVE = auto()
    NEGATIVE = auto()
class OracleAstNode(AstNode):
    def __init__(self, oracle_id, pole):
        self.oracle_id = oracle_id
        self.pole = pole

def print_ast(node, indent=0):
    if isinstance(node, OrAstNode):
        print(" " * indent + "OR")
        print_ast(node.left, indent + 2)
        print_ast(node.right, indent + 2)
    elif isinstance(node, SeqAstNode):
        print(" " * indent + "SEQ")
        print_ast(node.left, indent + 2)
        print_ast(node.right, indent + 2)
    elif isinstance(node, StarAstNode):
        print(" " * indent + "STAR")
        print_ast(node.left, indent + 2)
    elif isinstance(node, LiteralCharacterAstNode):
        print(" " * indent + f"LITERAL: {node.char!r}")
    elif isinstance(node, EmptySetAstNode):
        print (" " * indent + "EMPTY_SET: ∅")
    elif isinstance(node, EpsilonAstNode):
        print(" " * indent + "EPSILON: ε")
    elif isinstance(node, PositiveLookaheadAstNode):
        print(" " * indent + "POSITIVE_LOOKAHEAD")
        print_ast(node.left, indent + 2)
    elif isinstance(node, PositiveLookbehindAstNode):
        print(" " * indent + "POSITIVE_LOOKBEHIND")
        print_ast(node.left, indent + 2)
    elif isinstance(node, NegativeLookaheadAstNode):
        print(" " * indent + "NEGATIVE_LOOKAHEAD")
        print_ast(node.left, indent + 2)
    elif isinstance(node, NegativeLookbehindAstNode):
        print(" " * indent + "NEGATIVE_LOOKBEHIND")
        print_ast(node.left, indent + 2)
    elif isinstance(node, OracleAstNode):
        print(" " * indent + f"ORACLE: {node.oracle_id}")
    else:
        raise TypeError(f"Invalid AST node type: {type(node).__name__}")

In [3]:
from lark import Transformer, v_args

@v_args(inline=True)
class RegexAstTransformer(Transformer):
    def make_or(self, left, right):
        return OrAstNode(left, right)
    def make_seq(self, left, right):
        return SeqAstNode(left, right)
    def make_star(self, left):
        return StarAstNode(left)
    def group(self, left):
        return left
    def literal(self, token):
        return LiteralCharacterAstNode(str(token))
    def escaped(self, token):
        return LiteralCharacterAstNode(str(token)[1])
    def empty_set(self):
        return EmptySetAstNode()
    def epsilon(self):
        return EpsilonAstNode()
    def positive_lookahead(self, left):
        return PositiveLookaheadAstNode(left)
    def positive_lookbehind(self, left):
        return PositiveLookbehindAstNode(left)
    def negative_lookahead(self, left):
        return NegativeLookaheadAstNode(left)
    def negative_lookbehind(self, left):
        return NegativeLookbehindAstNode(left)
        


# build ast
* input: regex string
* output: ast

In [4]:
from lark import Lark
parser = Lark(
    REGEX_GRAMMAR,
    parser="lalr",
    start="start",
    transformer=RegexAstTransformer(),
)
def build_ast(regexString):
    ast = parser.parse(regexString)
    return ast
print_ast(build_ast("a|bc*"))

OR
  LITERAL: 'a'
  SEQ
    LITERAL: 'b'
    STAR
      LITERAL: 'c'


In [5]:
## definitoion of reversing regex
# rev(∅) -> ∅
# rev(ε) -> ε
# rev(a) -> a
# rev(r1 r2) -> rev(r2) rev(r1)
# rev(r1 | r2) -> rev(r1) | rev(r2)
# rev(r*) -> {rev(r)}*
# rev(oracle) -> oracle

def reverse_ast(node):
    if isinstance(node, (EmptySetAstNode, EpsilonAstNode, LiteralCharacterAstNode)):
        return node
    if isinstance(node, SeqAstNode):
        return SeqAstNode(reverse_ast(node.right), reverse_ast(node.left))
    if isinstance(node, OrAstNode):
        return OrAstNode(reverse_ast(node.left), reverse_ast(node.right))
    if isinstance(node, StarAstNode):
        return StarAstNode(reverse_ast(node.left))
    # look around
    if isinstance(node, OracleAstNode):
        return OracleAstNode(node.oracle_id, node.pole)
    raise TypeError(f"Invalid AST node type: {type(node).__name__}")

# Evaluate ast node with look around
1. Eval
   - input: direction, ast, text, {tape, next id}
   - output: oracle tape
2. EvalAux
   - input: node, text, {tape, next id}
   - output: node with OracleAstNode

In [6]:

class Direction(Enum):
    L2R = auto()
    R2L = auto()
class Context:
    def __init__(self):
        self.id = 0
        self.tapes = {}

def Eval(direction, ast, text, context):
    oracle_ast = EvalAux(ast, text, context)
    tape = Match(direction, oracle_ast, text, context.tapes)
    return tape

def EvalAux(node, text, context):
    if isinstance(node, (EmptySetAstNode, EpsilonAstNode, LiteralCharacterAstNode, OracleAstNode)):
        return node
    if isinstance(node, SeqAstNode):
        new_left = EvalAux(node.left, text, context)
        new_right = EvalAux(node.right, text, context)
        return SeqAstNode(new_left, new_right)
    if isinstance(node, OrAstNode):
        new_left = EvalAux(node.left, text, context)
        new_right = EvalAux(node.right, text, context)
        return OrAstNode(new_left, new_right)
    if isinstance(node, StarAstNode):
        new_left = EvalAux(node.left, text, context)
        return StarAstNode(new_left)
    # look ahead
    if isinstance(node, PositiveLookaheadAstNode):
        tape = Eval(Direction.R2L, node.left, text, context)
        oracle_id = context.id
        context.id += 1
        context.tapes[oracle_id] = tape
        return OracleAstNode(oracle_id, OraclePolarity.POSITIVE)
    # look behind
    if isinstance(node, PositiveLookbehindAstNode):
        tape = Eval(Direction.L2R, node.left, text, context)
        oracle_id = context.id
        context.id += 1
        context.tapes[oracle_id] = tape
        return OracleAstNode(oracle_id, OraclePolarity.POSITIVE)
    # nefative look ahead
    if isinstance(node, NegativeLookaheadAstNode):
        tape = Eval(Direction.R2L, node.left, text, context)
        oracle_id = context.id
        context.id += 1
        context.tapes[oracle_id] = tape
        return OracleAstNode(oracle_id, OraclePolarity.NEGATIVE)
    # negative look behind
    if isinstance(node, NegativeLookbehindAstNode):
        tape = Eval(Direction.L2R, node.left, text, context)
        oracle_id = context.id
        context.id += 1
        context.tapes[oracle_id] = tape
        return OracleAstNode(oracle_id, OraclePolarity.NEGATIVE)
    raise TypeError(f"Invalid AST node type: {type(node).__name__}")

# AST to NFA
* input: list of ast node
* output: NFA

In [7]:
class OracleEdge:
    def __init__(self, oracle_id, pole):
        self.oracle_id = oracle_id
        self.pole = pole
class NFA:
    def __init__(self, starting_state, final_state, transitions):
        self.starting_state = starting_state
        self.final_state = final_state
        self.transitions = transitions
        # transitions[状態][記号] = {遷移先の状態, ...}

    @property
    def states(self):
        # 状態の一覧を取り出す関数（遷移先で表れる）
        return set(self.transitions)

    #statesからε遷移で到達できる状態の集合
    def epsilon_closure(self, states):
        #found:= ε遷移だけで到達できる状態
        #まず, 出発地点の状態を登録
        found = set(states)
        stack = list(states)

        while stack:
            state = stack.pop()
            # 先頭の遷移で, 'None'で遷移できる状態を取り出し, reachable, stackに追加
            for nex in self.transitions[state].get(
                None, ()
            ):
                if nex not in found:
                    found.add(nex)
                    stack.append(nex)

        return found

    def move(self, states, char):
        """statesから文字charで遷移する先"""
        dest = set()
        for state in states:
            dest.update(self.transitions[state].get(char, ()))
        return dest

    def show(self):
        """状態と遷移を表示する。"""
        print(f"開始状態: q{self.starting_state}")
        print(f"受理状態: q{self.final_state}")
        
        for state in sorted(self.transitions):
            # .items()で(記号, 遷移先)のペアを取り出す
            for symbol, destinations in self.transitions[state].items():
                label = (
                    "ε"
                    if symbol is None
                    else repr(symbol)
                )
                # 遷移を表示する
                for destination in sorted(destinations):
                    print(
                        f"q{state} --{label}--> q{destination}"
                    )

In [8]:
class ThompsonConstruction:
    def __init__(self, ast):
        self.ast = ast

    def construct(self):
        # 呼び出すたびに新しいNFAを構築する。
        self.transitions = {}
        self.next_state = 0

        start, final = self._construct_from_ast(self.ast)

        return NFA(start, final, self.transitions)

    def _new_state(self):
        """新しい状態番号を発行する。"""
        state = self.next_state
        self.next_state += 1

        # この状態からの遷移は、最初は空。
        self.transitions[state] = {}

        return state

    def _add_edge(self, source, symbol, destination):
        """sourceからdestinationへの遷移を追加する。"""
        self.transitions[source].setdefault(symbol, set()).add(destination)
    def _construct_from_ast(self, node):
        """
        部分ASTに対応するNFAを作る。
        """
        if isinstance(node, EmptySetAstNode):
            start = self._new_state()
            final = self._new_state()
            return start, final
        if isinstance(node, EpsilonAstNode):
            start = self._new_state()
            final = self._new_state()
            self._add_edge(start, None, final)
            return start, final
        if isinstance(node, LiteralCharacterAstNode):
            if (
                not isinstance(node.char, str)
                or len(node.char) != 1
            ):
                raise ValueError(
                    "文字ノードには1文字の文字列を指定してください"
                )
            start = self._new_state()
            final = self._new_state()
            self._add_edge(start, node.char, final)
            return start, final
        # Oracle Ast Node
        if isinstance(node, OracleAstNode):
            start = self._new_state()
            final = self._new_state()
            self._add_edge(
                start, OracleEdge(node.oracle_id, node.pole), final
            )
            return start, final
        if isinstance(node, SeqAstNode):
            left_start, left_final = self._construct_from_ast(node.left)
            right_start, right_final = self._construct_from_ast(node.right)
            self._add_edge(left_final,None,right_start)
            return left_start, right_final
        if isinstance(node, OrAstNode):
            left_start, left_final = self._construct_from_ast(node.left)
            right_start, right_final = self._construct_from_ast(node.right)
            start = self._new_state()
            final = self._new_state()
            self._add_edge(start, None, left_start)
            self._add_edge(start, None, right_start)
            self._add_edge(left_final, None, final)
            self._add_edge(right_final, None, final)
            return start, final
        if isinstance(node, StarAstNode):
            sub_start, sub_final = (self._construct_from_ast(node.left))
            start = self._new_state()
            final = self._new_state()
            # 0回で終了する。
            self._add_edge(start, None, final)
            self._add_edge(start, None, sub_start)
            self._add_edge(sub_final, None, sub_start)
            self._add_edge(sub_final, None, final)
            return start, final
        raise TypeError(
            f"対応していないASTノード: {type(node).__name__}"
        )

# Matching of ONFA
1. Match
   オラクルノードを受け取り, 走査方向に別にSimulateを呼び, オラクルテープを返す。
    * input: direction, ast, text, oracle tape
    * output: oracle tape of ast in argument
3. SimulateL2R, SimulateR2L
   指定された方向に走査し, オラクルテープを構築する
    * input: nfa, text, tapes
    * output: oracle tape of nfa in argument

In [9]:
# Oracleエッジ付きのclousure
def oracle_closure(nfa, states, position, tapes):
    """ε遷移と、現在位置で真になるOracle遷移をたどる。"""
    found = set(states)
    stack = list(states)
    while stack:
        state = stack.pop()
        for symbol, nexs in nfa.transitions[state].items():
            # ε遷移は通過できる
            ok = symbol is None

            # オラクルエッジの場合, テーブルを参照する
            if isinstance(symbol, OracleEdge):
                if tapes[symbol.oracle_id] is False:
                    raise ValueError(f"Oracle {symbol.oracle_id} has not been evaluated")
                value = tapes[symbol.oracle_id][position]
                if symbol.pole == OraclePolarity.POSITIVE:
                    ok = value
                elif symbol.pole == OraclePolarity.NEGATIVE:
                    ok = not value
            if not ok:
                continue
            for nex in nexs:
                if nex not in found:
                    found.add(nex)
                    stack.append(nex)
    return found


def Match(direction, ast, text, tapes):
    if direction == Direction.R2L:
        ast = reverse_ast(ast)
    nfa = ThompsonConstruction(ast).construct()
    if direction == Direction.L2R:
        return SimulateL2R(nfa, text, tapes)
    if direction == Direction.R2L:
        return SimulateR2L(nfa, text, tapes)
    raise ValueError(
        f"Invalid direction: {direction!r}"
    )

def SimulateL2R(nfa, text, tapes):
    n = len(text)
    output = []
    output = [False] * (n + 1)
    currState = set()
    for pos in range(n + 1):
        currState.add(nfa.starting_state)
        currState = oracle_closure(nfa, currState, pos, tapes)
        if nfa.final_state in currState:
            output[pos] = True
        if pos < n:
            currState = nfa.move(currState, text[pos])
    return output
def SimulateR2L(nfa, text, tapes):
    n = len(text)
    output = []
    output = [False] * (n + 1)
    currState = set()
    for pos in range(n, -1, -1):
        currState.add(nfa.starting_state)
        currState = oracle_closure(nfa, currState, pos, tapes)
        if nfa.final_state in currState:
            output[pos] = True
        if pos > 0:
            currState = nfa.move(currState, text[pos - 1])
    return output

# find substring matching start_index
* input: regex_string, text
* output: list of maching start_index

In [10]:
def substring_match(regexStr, text):
    result = []
    n = len(text)
    ast = build_ast(regexStr)
    context = Context()
    main_tape = Eval(Direction.R2L, ast, text, context)
    for pos in range(n + 1):
        if main_tape[pos]:
            result.append(pos)

    return result

In [11]:
substring_match("(?<!cb)a(?!zz*)", "cbazzzzcbazgagz")

[12]